> 「构造词汇表」部分将介绍两种常见的子词分割方法：
>
> - **BPE（Byte-Pair Encoding）**：用于 GPT，GPT-2，RoBERTa，BART，GLM，LLaMA，Qwen 等模型。
> - **WordPiece**：用于 BERT、ALBERT、文心一言 等模型。
> - **T5/PaLM/Gemini/**：SentencePiece。
>
> 在线工具：[Tiktokenizer（推荐）](https://tiktokenizer.vercel.app) 



## 什么是 Tokenizer？

**Tokenizer**可以将文本转换为模型能够理解的数字ID序列，是自然语言和计算机语言交互的重要媒介。

### Encode阶段

1. **分词（Tokenize）**

   将文本拆分为Token，常见的分词方式包括字级、词级、子词级（如 BPE、WordPiece）等。

   ```sql
   输入: "丁师兄大模型"
   分词: ["丁", "师兄", "大", "模型"]
   ```

2. **映射（Mapping）**

   将每个Token映射为词汇表中的唯一ID，生成的序列即为模型的输入。

   ```sql
   分词: ["丁", "师兄", "大", "模型"]
   映射: [1001, 1002, 1003, 1004]
   ```

### Decode阶段

1. **反映射（De-mapping）**

   模型输出的ID序列通过词汇表映射回对应的token，这里是一一映射。

   ```sql
   输出: [1001, 1002, 1003, 1004]
   反映射: ["丁", "师兄", "大", "模型"]
   ```

2. **文本重组**

   将解码后的Token拼接为完整文本。

   ```sql
   反映射: ["丁", "师兄", "大", "模型"]
   重组: "丁师兄大模型"
   ```


# 环境依赖

In [1]:
! pip install transformers

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://bytedpypi.byted.org/simple/


In [2]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import transformers
print(transformers.__version__)

5.8.0


In [3]:
from transformers import AutoTokenizer
import os

# 设置 no_proxy 和 NO_PROXY
os.environ['no_proxy'] = "localhost,.byted.org,byted.org,.bytedance.net,bytedance.net,.byteintl.net,.tiktok-row.net,.tiktok-row.org,127.0.0.1,::1"
os.environ['NO_PROXY'] = os.environ['no_proxy']   # 保持两者一致

# 设置 Hugging Face 镜像端点
os.environ['HF_ENDPOINT'] = "http://huggingface-proxy-sg.byted.org"   # 或使用 https

# 现在再导入 transformers
from transformers import GPT2Model
# AutoTokenizer 是 Hugging Face 提供的一个自动选择分词器类的工厂。
tokenizer = AutoTokenizer.from_pretrained("gpt2")
# /home/tiger/.cache/huggingface/hub/models--gpt2/snapshots/607a30d783dfa663caf39e06633721c8d4cfcd7e 模型分词器位置
# config.json  模型配置文件
# merges.txt   merges.txt 决定 “怎么切” vocab.json 决定 “切完后的 token 编号是多少”
# tokenizer_config.json  分词器配置文件
# tokenizer.json 分词器文件
# vocab.json  词汇表编号文件


In [4]:
text = "hello, 大模型!"

# 编码
# 1.把文本变成tokens

tokens = tokenizer.tokenize(text)
# 这个会输出乱码，因为 GPT2 模型会把中文转换成 Unicode 编码的字符
# 所以，这里需要使用 Unicode 编码的字符
print("tokens: ", tokens)

# 2.把tokens转换成ID
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print("token ids: ", token_ids)


# 解码
# 3.把id转换成tokens
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print("tokens: ", tokens)

# 4.把tokens拼接成文本
decoded_text = tokenizer.convert_tokens_to_string(tokens)
print("decoded text: ", decoded_text)

tokens:  ['hello', ',', 'Ġå¤', '§', 'æ', '¨', '¡', 'å', 'ŀ', 'ĭ', '!']
token ids:  [31373, 11, 36469, 100, 162, 101, 94, 161, 252, 233, 0]
tokens:  ['hello', ',', 'Ġå¤', '§', 'æ', '¨', '¡', 'å', 'ŀ', 'ĭ', '!']
decoded text:  hello, 大模型!


In [5]:
for token in tokens:
    byte_sequence = list(token.encode("utf-8"))
    byte_str = bytes(byte_sequence).decode("utf-8", errors="replace")
    print(f"Token: '{token}' -> Bytes: '{byte_sequence}' -> Decoded: '{byte_str}'")

Token: 'hello' -> Bytes: '[104, 101, 108, 108, 111]' -> Decoded: 'hello'
Token: ',' -> Bytes: '[44]' -> Decoded: ','
Token: 'Ġå¤' -> Bytes: '[196, 160, 195, 165, 194, 164]' -> Decoded: 'Ġå¤'
Token: '§' -> Bytes: '[194, 167]' -> Decoded: '§'
Token: 'æ' -> Bytes: '[195, 166]' -> Decoded: 'æ'
Token: '¨' -> Bytes: '[194, 168]' -> Decoded: '¨'
Token: '¡' -> Bytes: '[194, 161]' -> Decoded: '¡'
Token: 'å' -> Bytes: '[195, 165]' -> Decoded: 'å'
Token: 'ŀ' -> Bytes: '[197, 128]' -> Decoded: 'ŀ'
Token: 'ĭ' -> Bytes: '[196, 173]' -> Decoded: 'ĭ'
Token: '!' -> Bytes: '[33]' -> Decoded: '!'


Token: 'hello' -> Bytes: '[104, 101, 108, 108, 111]' -> Decoded: 'hello' // h->104 e->101 l->108 o->111
Token: ',' -> Bytes: '[44]' -> Decoded: ','
Token: 'Ġå¤' -> Bytes: '[196, 160, 195, 165, 194, 164]' -> Decoded: 'Ġå¤'
Token: '§' -> Bytes: '[194, 167]' -> Decoded: '§'
Token: 'æ' -> Bytes: '[195, 166]' -> Decoded: 'æ'
Token: '¨' -> Bytes: '[194, 168]' -> Decoded: '¨'
Token: '¡' -> Bytes: '[194, 161]' -> Decoded: '¡'
Token: 'å' -> Bytes: '[195, 165]' -> Decoded: 'å'
Token: 'ŀ' -> Bytes: '[197, 128]' -> Decoded: 'ŀ'
Token: 'ĭ' -> Bytes: '[196, 173]' -> Decoded: 'ĭ'
Token: '!' -> Bytes: '[33]' -> Decoded: '!'

# WordPiece分词器

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") # 中文不好切分会失败

text = "hello, LLM!"

# 编码
# 1.把文本变成tokens

tokens = tokenizer.tokenize(text)
print("tokens: ", tokens)

# 2.把tokens转换成ID
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print("token ids: ", token_ids)


# 解码
# 3.把id转换成tokens
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print("tokens: ", tokens)

# 4.把tokens拼接成文本
decoded_text = tokenizer.convert_tokens_to_string(tokens)
print("decoded text: ", decoded_text)

tokens:  ['hello', ',', 'll', '##m', '!']
token ids:  [7592, 1010, 2222, 2213, 999]
tokens:  ['hello', ',', 'll', '##m', '!']
decoded text:  hello, llm!


# encode和decode方法

In [7]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "hello, 大模型!"

token_ids = tokenizer.encode(text)
print("token ids: ", token_ids)

token ids:  [31373, 11, 36469, 100, 162, 101, 94, 161, 252, 233, 0]


In [8]:
decoded_text = tokenizer.decode(token_ids)
print("decoded text: ", decoded_text)

decoded text:  hello, 大模型!


# Tokenizer的基础属性

In [9]:
tokenizer.vocab_size # 词汇表大小

50257

In [10]:
tokenizer.convert_tokens_to_ids("Ġ") # 空格和下一个词会被合并 Ġhello

220

In [11]:
tokenizer.all_special_tokens # 所有特殊token

['<|endoftext|>']

In [12]:
tokenizer.special_tokens_map # 特殊token映射
"""
本质
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

print(tokenizer.bos_token)  # '<|endoftext|>'
print(tokenizer.eos_token)  # '<|endoftext|>'
print(tokenizer.unk_token)  # '<|endoftext|>'
三个属性，指向同一个字符串，对应同一个 id（50256）
"""

'\n本质\nfrom transformers import AutoTokenizer\ntokenizer = AutoTokenizer.from_pretrained("gpt2")\n\nprint(tokenizer.bos_token)  # \'<|endoftext|>\'\nprint(tokenizer.eos_token)  # \'<|endoftext|>\'\nprint(tokenizer.unk_token)  # \'<|endoftext|>\'\n三个属性，指向同一个字符串，对应同一个 id（50256）\n'

In [13]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

In [14]:
tokenizer.vocab_size # 词汇表比gpt2大很多

151643

In [15]:
tokenizer.all_special_tokens

['<|im_end|>',
 '<|endoftext|>',
 '<|im_start|>',
 '<|object_ref_start|>',
 '<|object_ref_end|>',
 '<|box_start|>',
 '<|box_end|>',
 '<|quad_start|>',
 '<|quad_end|>',
 '<|vision_start|>',
 '<|vision_end|>',
 '<|vision_pad|>',
 '<|image_pad|>',
 '<|video_pad|>']

In [16]:
tokenizer.special_tokens_map

{'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}

# BPE实现细节

## 分词

我们需要将语料库的文本拆分为单词，假设当前语料库包含的单词和对应频次如下：

```sql
("low", 5), ("lower", 2), ("newest", 6), ("widest", 3)
```

### 构造词汇表

#### Byte-Pair Encoding (BPE)

BPE 每次的迭代目标是找到频率最高的相邻字符对，定义 Score：

$$
\text{Score}_{\text{BPE}}(x, y) = \text{freq}(x, y)
$$
其中，$\text{freq}(x, y)$ 表示字符对 $(x, y)$ 在语料库中的出现频次。

##### 步骤

1. **初始化词汇表 $V$**：
   - $V$ 包含语料库中的所有唯一字符，即单词字符的集合。
2. **统计字符对的频次**：
   - 对于每个单词的字符序列，统计相邻字符对的出现频次。
3. **找到频次最高的字符对并合并**：
   - 选择出现频率最高的字符对 $(x, y)$，将其合并为新符号 $xy$。
4. **更新词汇表并重复步骤 2 到 4**：
   - 将新符号添加到词汇表 $V = V \cup \{xy\}$。
   - 更新语料库中的单词表示，重复统计和合并过程，直到满足停止条件（例如，词汇表达到预定大小）。

##### 示例

**步骤 1：初始化词汇表**

- **将单词拆分为字符序列**：

  ```plaintext
  ("l", "o", "w"), 5  
  ("l", "o", "w", "e", "r"), 2  
  ("n", "e", "w", "e", "s", "t"), 6  
  ("w", "i", "d", "e", "s", "t"), 3
  ```

- **词汇表 $V$**：

  ```plaintext
  {'l', 'o', 'w', 'e', 'r', 'n', 's', 't', 'i', 'd'}
  ```

**步骤 2：统计字符对的频次**

编写一个函数，根据给定的单词和其频次，自动统计字符对的频次。

In [17]:
from collections import defaultdict

def count_char_pairs(word_freq):
    '''统计字符对的频次'''
    pair_freq = defaultdict(int)
    for word, freq in word_freq:
        chars = list(word)
        for i in range(len(chars) - 1):
            pair = (chars[i], chars[i + 1])
            pair_freq[pair] += freq
    return pair_freq

word_freq = [
    ("low", 5),
    ("lower", 2),
    ("newest", 6),
    ("widest", 3),
]

pair_freq = count_char_pairs(word_freq)

for pair, freq in pair_freq.items():
    print(f"{pair}: {freq}")

('l', 'o'): 7
('o', 'w'): 7
('w', 'e'): 8
('e', 'r'): 2
('n', 'e'): 6
('e', 'w'): 6
('e', 's'): 9
('s', 't'): 9
('w', 'i'): 3
('i', 'd'): 3
('d', 'e'): 3


**步骤 3：找到频次最高的字符对并合并**

- **选择频次最高的字符对**：

  - `("e", "s")` 和 `("s", "t")`，频次均为 9。可以任选其一进行合并，假设选择排序第一的： `("e", "s")`。

- **合并 `("e", "s")` 为新符号 `es`**。

- **记录合并操作**：

  ```plaintext
  Merge 1: ("e", "s") -> "es"
  ```

**步骤 4：更新词汇表并重复**

- **更新单词序列**：

  ```plaintext
  ("l", "o", "w"), 5  
  ("l", "o", "w", "e", "r"), 2  
  ("n", "e", "w", "es", "t"), 6  
  ("w", "i", "d", "es", "t"), 3
  ```

- **更新词汇表 $V$**：

  ```plaintext
  {'l', 'o', 'w', 'e', 'r', 'n', 's', 't', 'i', 'd', 'es'}
  ```

- **重复步骤 2 到 4，直到达到预定的词汇表大小**。

In [18]:
from collections import defaultdict

def find_best_pair(freq):
    """找到频次最高的字符对"""
    if not freq:
        return None, 0
    # 类似c++的max_element key=freq.get 是一个函数指针，用于获取每个字符对的频次
    # max() 函数会返回频次最高的字符对
    best_pair = max(freq, key=freq.get)
    return best_pair, freq[best_pair]

def merge_pair(word_freq, pair_to_merge):
    '''合并指定的字符对并添加到vocab'''
    merged_word_freq = []
    pair_str = ''.join(pair_to_merge)
    for word, freq in word_freq:
        new_word = []
        i = 0
        while i < len(word):
            # 检查当前字符和下一个字符是否是要合并的字符对
            if i < len(word) - 1 and word[i] == pair_to_merge[0] and word[i + 1] == pair_to_merge[1]:
                new_word.append(pair_str)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        merged_word_freq.append((new_word, freq))
    return merged_word_freq

def bpe_merge(word_freq, vocab_size):
    """执行BPE的合并操作，直到词汇表达到预定义的大小"""

    # 初始化词表
    vocab = set()
    for word, _ in word_freq:
        vocab.update(word)

    # 记录要合并的merges
    merges = []
    while len(vocab) < vocab_size:
        pair_freq = count_char_pairs(word_freq)
        best_pair, best_freq = find_best_pair(pair_freq)
        if not best_pair:
            break
        word_freq = merge_pair(word_freq, best_pair)
        new_symbol = ''.join(best_pair)
        vocab.add(new_symbol)
        merges.append((best_pair, new_symbol))
        print(f"Merge: {best_pair} -> {new_symbol}, 词汇表大小: {len(vocab)}")

    return vocab, merges

# 初始化的语料
word_freq = [
    (['l', 'o', 'w'], 5),
    (['l', 'o', 'w', 'e', 'r'], 2),
    (['n', 'e', 'w', 'e', 's', 't'], 6),
    (['w', 'i', 'd', 'e', 's', 't'], 3)
]

# 预定义的词汇表大小
vocab_size = 13

final_vocab, merge_records = bpe_merge(word_freq, vocab_size)

print("\n最终词汇表 V: ")
print(final_vocab)

print("\n合并记录: ")
for idx, (pair, new_sym) in enumerate(merge_records, 1):
    print(f"Merge {idx}: {pair} -> {new_sym}")

Merge: ('e', 's') -> es, 词汇表大小: 11
Merge: ('es', 't') -> est, 词汇表大小: 12
Merge: ('l', 'o') -> lo, 词汇表大小: 13

最终词汇表 V: 
{'d', 'o', 'w', 'i', 'e', 'es', 'r', 's', 'l', 't', 'lo', 'est', 'n'}

合并记录: 
Merge 1: ('e', 's') -> es
Merge 2: ('es', 't') -> est
Merge 3: ('l', 'o') -> lo


## 映射（Mapping）

以 BPE 为例，根据最终词汇表 $V$ ，简单实现 Token 和 ID 之间的映射关系的代码：

In [19]:
token_to_id = {token: idx for idx, token in enumerate(final_vocab)}

id_to_token = {idx: token for token, idx in token_to_id.items()}

print("Token to ID:", token_to_id)
print("ID to Token:", id_to_token)

Token to ID: {'d': 0, 'o': 1, 'w': 2, 'i': 3, 'e': 4, 'es': 5, 'r': 6, 's': 7, 'l': 8, 't': 9, 'lo': 10, 'est': 11, 'n': 12}
ID to Token: {0: 'd', 1: 'o', 2: 'w', 3: 'i', 4: 'e', 5: 'es', 6: 'r', 7: 's', 8: 'l', 9: 't', 10: 'lo', 11: 'est', 12: 'n'}


## 预分词（Pre-tokenization）原理

**预分词**是子词分词流程（BPE / WordPiece / Unigram）的**第一道工序**，作用是把一段连续的原始字符串先按"明显的语义/字符边界"切成一组**粗粒度的"词单元"**，再交给后续的子词算法在每个单元内部做细粒度合并。它存在的根本原因有两个：一是**防止跨语义边界的合并** —— 如果直接对整段文本跑 BPE，算法会发现 `"the_"`、`"_of_"` 这种高频组合并把它们合成一个 token，破坏词与词之间的独立性；二是**控制算法复杂度** —— BPE 每轮要统计所有相邻字符对的频率，复杂度是 O(N²)，N 是序列长度，预切分后每个"词"通常只有几到十几个字符，统计量从平方级降到线性级。常见策略有四类：**Whitespace** 按空格 + 标点切（BERT 风格），**ByteLevel** 把文本先编码成 UTF-8 字节、再把空格替换成可见字符 `Ġ`（GPT-2/RoBERTa 风格，能无损还原任意 Unicode），**Metaspace** 把空格替换成 `▁` 后整体看作前缀（SentencePiece/LLaMA 风格，对中日韩无空格语言友好），**BertPreTokenizer** 在 Whitespace 基础上额外做小写化和重音去除。Hugging Face 的预分词器返回的不是单纯的字符串列表，而是 `[(word, (start, end)), ...]` 这种带**字符偏移量**的元组 —— 偏移量在训练阶段可以丢弃，但在推理阶段（特别是 NER、问答、span 抽取）必须保留，因为模型最终要把预测的 token span 映射回原文的字符位置。一句话总结：**预分词 = 用确定性规则做粗切，BPE/WordPiece = 用统计方法做细切**，两者配合既保住了语义边界，又把"哪几个字符组合成一个 token"这个决策权留给了数据驱动的学习过程。

# 编码应用

In [20]:
def tokenize(text):
    """新文本的编码操作"""

    # 预分词
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]

    print("开始预分词的结果：")
    print(pre_tokenized_text)

    splits = [[l for l in word] for word in pre_tokenized_text]
    print("\n开始拆分的结果：")
    print(splits)

    # 合并
    # 遍历每一条merges
    for pair, merge in merges.items():
        print(f"\n应用合并规则：{pair} -> {merge}")

        # 遍历每一条splits: char array
        for idx, split in enumerate(splits):
            print(f"    合并前的{idx+1}个单词: {split}")
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                    print(f"    在位置 {i} 处合并: {split}")
                else:
                    i += 1
            splits[idx] = split

    print("\n最终的拆分结果: ")
    print(splits)

    
    tokens = sum(splits, [])
    print("\n最终生成的 Tokens: ")
    print(tokens)
    
    token_ids = [token_to_id.get(token, '[UNK]') for token in tokens]
    print("\n最终的编码：Token IDs: ")
    print(token_ids)

    return token_ids
    

merges = {
    ('e', 's'): 'es',
    ('es', 't'): 'est',
    ('l', 'o'): 'lo'
}

text = "lower,estimate"

token_ids = tokenize(text)

开始预分词的结果：
['lower', ',estimate']

开始拆分的结果：
[['l', 'o', 'w', 'e', 'r'], [',', 'e', 's', 't', 'i', 'm', 'a', 't', 'e']]

应用合并规则：('e', 's') -> es
    合并前的1个单词: ['l', 'o', 'w', 'e', 'r']
    合并前的2个单词: [',', 'e', 's', 't', 'i', 'm', 'a', 't', 'e']
    在位置 1 处合并: [',', 'es', 't', 'i', 'm', 'a', 't', 'e']

应用合并规则：('es', 't') -> est
    合并前的1个单词: ['l', 'o', 'w', 'e', 'r']
    合并前的2个单词: [',', 'es', 't', 'i', 'm', 'a', 't', 'e']
    在位置 1 处合并: [',', 'est', 'i', 'm', 'a', 't', 'e']

应用合并规则：('l', 'o') -> lo
    合并前的1个单词: ['l', 'o', 'w', 'e', 'r']
    在位置 0 处合并: ['lo', 'w', 'e', 'r']
    合并前的2个单词: [',', 'est', 'i', 'm', 'a', 't', 'e']

最终的拆分结果: 
[['lo', 'w', 'e', 'r'], [',', 'est', 'i', 'm', 'a', 't', 'e']]

最终生成的 Tokens: 
['lo', 'w', 'e', 'r', ',', 'est', 'i', 'm', 'a', 't', 'e']

最终的编码：Token IDs: 
[10, 2, 4, 6, '[UNK]', 11, 3, '[UNK]', '[UNK]', 9, 4]
